In [4]:
import glob
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 50)

path = glob.glob("data/public_reviews_dataset_cleaned.csv")[0]
df = pd.read_csv(path, low_memory=False)
print(path, df.shape)
print(df.dtypes)
df.head(3)

COL = { 
    "pid": "asin",
    "rid": "reviewer_id",
    "rating": "review_rating",
    "text": "review_text",
    "date": "review_date",
}
LABEL = "fake_review_product"
LEAK = ["reviewer_classified_fake", "reviewer_classified_honest",
         "reviewer_labeled_fake", "reviewer_labeled_honest"]
CAMP = "fake_review_campaign_start_date"
DEL = "review_is_removed_by_amazon"

df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce")

data/public_reviews_dataset_cleaned.csv (381734, 23)
asin                                   str
review_id                              str
reviewer_id                            str
review_title                           str
review_text                            str
review_rating                        int64
review_date                            str
product_title                          str
product_url                            str
number_of_helpful                  float64
number_of_photos                   float64
photo_thumbnail_urls                   str
photo_fullsize_urls                    str
asin_url                               str
review_url                             str
reviewer_url                           str
fake_review_campaign_start_date        str
fake_review_product                   bool
reviewer_classified_fake              bool
reviewer_classified_honest            bool
reviewer_labeled_fake              float64
reviewer_labeled_honest            float64
r

In [5]:
pid, rat, txt, dt = COL["pid"], COL["rating"], COL["text"], COL["date"]

print("yorum:", len(df), " ürün:", df[pid].nunique(), " yorumcu:", df[COL["rid"]].nunique())
print("tarih aralığı:", df[dt].min(), "->", df[dt].max())

# Etiket dağılımı (yorum düzeyinde ve ürün düzeyinde)
print(df[LABEL].value_counts(dropna=False))
print(df.groupby(pid)[LABEL].first().value_counts(dropna=False))

# Etiket bir ürün içinde sabit mi? (0 çıkmalı)
chk = df.groupby(pid)[LABEL].nunique()
print("etiketi değişen ürün sayısı:", int((chk > 1).sum()))

# Puan dağılımı ve ürün başına yorum sayısı
print(df[rat].value_counts(normalize=True).sort_index().round(3))
print(df.groupby(pid).size().describe().round(1))

# Etiketle ilgili sütunlar: ne kadarı boş, kaç farklı değer var?
for c in LEAK + [CAMP, DEL]:
    print(c, "| boş oranı:", round(df[c].isna().mean(), 3), "| farklı değer:", df[c].nunique(dropna=True))

# Silinme sütunu 15 Haziran 2020'den sonra güvenilmez:
cut = pd.Timestamp("2020-06-15")
print("15 Haz 2020 sonrası yorum oranı:", round((df[dt] >= cut).mean(), 3))

yorum: 381734  ürün: 3389  yorumcu: 334342
tarih aralığı: 2000-06-12 00:00:00 -> 2021-01-19 00:00:00
fake_review_product
False    221181
True     160553
Name: count, dtype: int64
fake_review_product
False    1895
True     1494
Name: count, dtype: int64
etiketi değişen ürün sayısı: 0
review_rating
1    0.289
2    0.117
3    0.140
4    0.278
5    0.176
Name: proportion, dtype: float64
count    3389.0
mean      112.6
std       184.4
min         1.0
25%        17.0
50%        46.0
75%       127.0
max      2251.0
dtype: float64
reviewer_classified_fake | boş oranı: 0.0 | farklı değer: 2
reviewer_classified_honest | boş oranı: 0.0 | farklı değer: 2
reviewer_labeled_fake | boş oranı: 0.79 | farklı değer: 2
reviewer_labeled_honest | boş oranı: 0.79 | farklı değer: 2
fake_review_campaign_start_date | boş oranı: 0.624 | farklı değer: 206
review_is_removed_by_amazon | boş oranı: 0.226 | farklı değer: 2
15 Haz 2020 sonrası yorum oranı: 0.228


In [6]:
g = df.groupby(COL["pid"]).agg(kampanya_var=(CAMP, lambda s: s.notna().any()), etiket=(LABEL, "first"))
pd.crosstab(g["kampanya_var"], g["etiket"])

etiket,False,True
kampanya_var,,
False,1895,45
True,0,1449
